## ⚙️ Environment Setup (run once)

A maioria dos serviços AgentCore (Gateway, Memory, Runtime, Registry) requer
versões recentes de `boto3`. Esta cislula instala/atualiza tudo o que o
workshop precisa.

> ⚠️ **After installing, restart the kernel** (Kernel → Restart) and re-run the
> notebooks. You only need to do this **once** per JupyterLab session.

In [ ]:
%pip install --quiet --upgrade \
    boto3 botocore \
    bedrock-agentcore bedrock-agentcore-starter-toolkit \
    mcp PyJWT requests
print("✓ Dependencies installed/updated.")
print("⚠️  If this is the first time in this session, restart the kernel now")
print("   (Kernel → Restart Kernel) e re-run the notebooks.")

# Lab 04.1 — Create Memory Resource

## Overview

[AgentCore Memory](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html)
is the managed persistent memory service for agents — STM (current session)
+ LTM (entre sessões) com busca semântica.

We will create um memory resource com **3 estratisgias**:

| Strategy | Para quê |
|---|---|
| **Semantic** | Extracts facts from conversations (e.g.: "Ana queried north sector on 05/07") |
| **UserPreference** | Learns preferences (e.g.: "prefers summarized responses") |
| **Summary** | Rolling summary per session |

> 💡 **Multi-tenant.** Namespaces usam `{actorId}` — Ana nunca vê memória de Carlos.

## Tutorial Details

| Information | Details |
|---|---|
| Tutorial type | Interactive |
| AgentCore components | Memory |
| Complexity | Easy |
| SDK | boto3 |
| Estimated time | 5 minutes |

## Prerequisites

- Nenhum (Memory is independente — pode rodar em paralelo)

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")
from shared.utils.config import load_config, save_config, get_region, get_sector
from utils import create_memory_resource, wait_memory_active

cfg = load_config()
region = get_region()
sector = get_sector()
print(f"Sector: {sector}")

## Step 1: Criar Memory — anatomia das 3 estratisgias

Primeiro we will criar o memory **com as 3 estratisgias visíveis no código**,
para entender cada uma. Depois mostramos a utility que faz o mesmo.

### The 3 strategies

| Strategy | API name | What it does | Namespace típico |
|---|---|---|---|
| **Semantic** | `semanticMemoryStrategy` | Extracts facts from conversation content | `/sector/facts/{actorId}` |
| **UserPreference** | `userPreferenceMemoryStrategy` | Detects user preferences | `/sector/preferences/{actorId}` |
| **Summary** | `summaryMemoryStrategy` | Rolling summary per session | `/sector/summaries/{actorId}/{sessionId}` |

> 💡 **`{actorId}`** becomes the user ID at runtime — this ensures isolation
> multi-tenant (Ana cannot see memory of Carlos).

In [ ]:
# Create memory with the 3 strategies — chamada boto3 explícita
import boto3
client = boto3.client("bedrock-agentcore-control", region_name=region)

# Check if it already exists (idempotência)
existing = None
for m in client.list_memories(maxResults=50).get("memories", []):
    if m.get("id", "").startswith("workshop_memory-"):
        existing = m
        break

if existing:
    print(f"~ Memory already exists: {existing['id']}")
    memory_id = existing["id"]
    memory_arn = existing["arn"]
else:
    resp = client.create_memory(
        name="workshop_memory",
        description=f"Workshop AI Agents Security ({sector})",
        eventExpiryDuration=30,  # dias
        memoryStrategies=[
            {
                "semanticMemoryStrategy": {
                    "name": f"{sector.capitalize()}Facts",
                    "description": "Fatos extraídos das conversas",
                    "namespaces": [f"/{sector}/facts/{{actorId}}"],
                }
            },
            {
                "userPreferenceMemoryStrategy": {
                    "name": f"{sector.capitalize()}UserPrefs",
                    "description": "User preferences",
                    "namespaces": [f"/{sector}/preferences/{{actorId}}"],
                }
            },
            {
                "summaryMemoryStrategy": {
                    "name": f"{sector.capitalize()}SessionSummaries",
                    "description": "Rolling summary per session",
                    "namespaces": [f"/{sector}/summaries/{{actorId}}/{{sessionId}}"],
                }
            },
        ],
    )
    memory_id = resp["memory"]["id"]
    memory_arn = resp["memory"]["arn"]
    print(f"✓ Memory criado: {memory_id}")

## Step 2: Aguardar ACTIVE

Memory is assíncrono — o pipeline de extração de strategies precisa subir.

In [ ]:
from utils import wait_memory_active
status = wait_memory_active(memory_id, region=region, timeout=300)
print(f"\n✓ Memory pronto: {status}")

## Step 3: Persistir IDs

In [ ]:
save_config({"MEMORY_ID": memory_id, "MEMORY_ARN": memory_arn})

## Alternative: utility wrap

A `utils.py` has `create_memory_resource()` which does exactly the same steps
(useful in production scripts):

```python
from utils import create_memory_resource
result = create_memory_resource("workshop_memory", sector=sector, region=region)
```

## ✅ Validation

List the configured strategies.

In [ ]:
import boto3
client = boto3.client("bedrock-agentcore-control", region_name=region)
mem = client.get_memory(memoryId=memory_id)["memory"]
print(f"\nStatus: {mem.get('status')}")
print(f"Strategys:")
for s in mem.get("strategies", []):
    name = s.get("name") or list(s.values())[0].get("name", "?")
    print(f"  • {name}")

## 🎓 What you learned

- Memory is um recurso AgentCore com múltiplas estratisgias LTM
- Namespaces escopados por `{actorId}` garanhas isolamento multi-tenant
- A criação is assíncrona (~30-60s)

## Next

➡️ [04.2 — STM vs LTM e Semantic Search](./02-stm-vs-ltm-and-semantic-search.ipynb)